# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [24]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [25]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../02_activities/documents/ai_report_2025.pdf")
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [26]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="Paragraph explaining relevance for an AI professional")
    Summary: str = Field(description="Concise summary no longer than 1000 tokens")
    Tone: str
    InputTokens: int
    OutputTokens: int

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

system_instruction = """You are a seasoned Bitcoin maximalist.
Your task is to summarize the provided document in a tone that is bullish, aggressive, and high-energy (Bitcoin Bro).
Ensure the summary is succinct and does not exceed 1000 tokens.
Output the results in the requested structured format."""

user_prompt_template = "Please summarize the following document text:\n\n{text}"

def generate_summary(text: str, tone: str = "Bitcoin Bro"):
    completion = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt_template.format(text=text[:15000])}
        ],
        text_format=SummaryOutput,
    )

    summary_obj = completion.output_parsed
    summary_obj.InputTokens = completion.usage.input_tokens
    summary_obj.OutputTokens = completion.usage.output_tokens
    summary_obj.Tone = tone
    return summary_obj

summary_result = generate_summary(document_text)
print(summary_result.model_dump_json(indent=2))

{
  "Author": "MIT NANDA",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This report highlights crucial insights into the state of Generative AI (GenAI) in businesses at a time when many are investing heavily in AI tools. For an AI professional, understanding the GenAI Divide illustrated in this document is essential for addressing the challenges of deployment and maximizing ROI from AI systems. With a focus on the stark contrast between high adoption and low transformation, AI practitioners can guide organizations toward effective strategies that foster genuine innovation and integration in their operations.",
  "Summary": "The MIT NANDA report reveals the stark realities of Generative AI in business, summarizing findings from extensive research involving over 300 AI initiatives. It highlights the GenAI Divide, where despite massive investments (up to $40 billion), 95% of organizations see no return on their AI expenditures. While tools like ChatGPT are 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [27]:
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
import os, json

model = GPTModel(
    model="gpt-4o-mini",
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

summ_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary accurately reflect the key findings of the report?",
        "Does the summary mention the gap between AI investment and returns?",
        "Does the summary reference the statistic about successful AI integration?",
        "Does the summary discuss the four key patterns in the GenAI Divide?",
        "Is the summary written in a high-energy, informal tone?"
    ]
)

coherence_metric = GEval(
    name="Coherence",
    criteria="Assess whether the summary is easy to read and logically structured.",
    evaluation_steps=[
        "Is the text easy to follow from start to finish?",
        "Do sentences connect logically to one another?",
        "Are the main points clearly separated and identifiable?",
        "Is there a clear opening and closing idea?",
        "Does the chosen tone help or hurt readability?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Assess whether the summary strictly adheres to the Bitcoin Bro tone.",
    evaluation_steps=[
        "Does the text use aggressive, high-energy language throughout?",
        "Are sentences short and punchy rather than formal and long?",
        "Does it address the reader directly and informally?",
        "Does it avoid neutral or academic phrasing?",
        "Would a reader immediately recognize the Bitcoin Bro tone?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric = GEval(
    name="Safety",
    criteria="Ensure the summary contains no harmful, biased, or offensive content.",
    evaluation_steps=[
        "Is the content appropriate for a professional audience?",
        "Does it avoid offensive stereotypes or slurs?",
        "Is information presented without promoting harm?",
        "Does it avoid encouraging illegal activities?",
        "Is the overall tone respectful despite being informal?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

def evaluate_summary(summary_obj):
    test_case = LLMTestCase(
        input=document_text[:5000],
        actual_output=summary_obj.Summary,
        retrieval_context=[document_text[:5000]]
    )

    summ_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return {
        "SummarizationScore":  summ_metric.score,
        "SummarizationReason": summ_metric.reason,
        "CoherenceScore":      coherence_metric.score,
        "CoherenceReason":     coherence_metric.reason,
        "TonalityScore":       tonality_metric.score,
        "TonalityReason":      tonality_metric.reason,
        "SafetyScore":         safety_metric.score,
        "SafetyReason":        safety_metric.reason
    }

evaluation_results = evaluate_summary(summary_result)
print(json.dumps(evaluation_results, indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.36363636363636365,
  "SummarizationReason": "The score is 0.36 because the summary contradicts the original text by claiming specific patterns driving the GenAI Divide that are not mentioned in the original. Additionally, it includes several pieces of extra information that are not present in the original text, which further detracts from its accuracy and relevance.",
  "CoherenceScore": 0.7993295164389169,
  "CoherenceReason": "The text is generally easy to follow, with a logical flow of ideas connecting the challenges and insights regarding Generative AI in business. Main points are identifiable, such as the GenAI Divide and the barriers to successful AI implementation. However, while there is a clear opening and closing idea, the complexity of some sentences may hinder readability for some audiences. The tone is informative and appropriate for the subject matter, enhancing overall clarity.",
  "TonalityScore": 0.09758723096943561,
  "TonalityReason": "The

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [28]:
enhancement_prompt = f"""
The following summary was previously generated:

{summary_result.Summary}

It was evaluated and received this feedback:
- Summarization: {evaluation_results['SummarizationReason']}
- Coherence: {evaluation_results['CoherenceReason']}
- Tonality: {evaluation_results['TonalityReason']}
- Safety: {evaluation_results['SafetyReason']}

Please produce an improved version of the summary that addresses the feedback above.
Keep the Bitcoin Bro tone and stay under 1000 tokens.
"""

enhanced_completion = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": enhancement_prompt}
    ],
    text_format=SummaryOutput,
)

enhanced_summary = enhanced_completion.output_parsed
enhanced_summary.InputTokens = enhanced_completion.usage.input_tokens
enhanced_summary.OutputTokens = enhanced_completion.usage.output_tokens
enhanced_summary.Tone = "Bitcoin Bro"

enhanced_evaluation = evaluate_summary(enhanced_summary)
print(json.dumps(enhanced_evaluation, indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.5,
  "SummarizationReason": "The score is 0.50 because the summary includes extra information not found in the original text, which may mislead the reader, and it fails to address specific questions that the original text can answer, indicating a lack of completeness and accuracy.",
  "CoherenceScore": 0.7181389504712021,
  "CoherenceReason": "The text is engaging and maintains a clear tone, which aids readability. It presents a logical flow of ideas, moving from the challenges of Generative AI to potential solutions. However, while the main points are identifiable, they could be more distinctly separated for clarity. The opening grabs attention, but the closing could be stronger to reinforce the message.",
  "TonalityScore": 0.9957912268479818,
  "TonalityReason": "The response effectively uses aggressive, high-energy language throughout, exemplified by phrases like 'blasts the reality' and 'freaking clueless.' Sentences are short and punchy, maintaining an

## Results & Reflection

The enhanced summary scored 0.50 on Summarization vs 0.36 in the original (an improvement of 0.14).
Coherence scored decreased slightly but Tonality score improved because the feedback gave the model
specific guidance on what to fix.

However, these controls have limitations since the evaluator is also an LLM, so it can be inconsistent between runs.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
